# 🍵 Tea Leaf Disease Detection – Deep Learning Classifier

This notebook builds a **convolutional neural network** to classify tea leaf diseases into **7 categories** using **transfer learning**.

- Dataset: TeaLeafBD (Mendeley Data)
- Total images: 5,276
- 7 disease categories (+ healthy leaves)
- Goal: Accurate classification of unseen test images

Main techniques:
- Transfer Learning (EfficientNet, MobileNetV2)
- Data Augmentation
- Class Weights balancing
- Fine-Tuning
- Focal Loss for difficult samples

## 🔧 Imports & Environment Setup

In [2]:
# Install required packages (useful for Colab or first runs)
%pip install -q tensorflow matplotlib pandas

# -------------------------------
# 🔧 Core Python & Data Handling
# -------------------------------
import os                    # OS utilities for file/directory management
import numpy as np           # Numerical operations on arrays
import pandas as pd          # Data manipulation (used later for submission files)
from collections import Counter  # For inspecting class distributions

# -------------------------------
# 📊 Visualization
# -------------------------------
import matplotlib.pyplot as plt  # For plotting training metrics and inspecting data

# -------------------------------
# 🤖 TensorFlow & Keras Components
# -------------------------------
import tensorflow as tf  # TensorFlow backend for deep learning models

# Data augmentation utilities
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Transfer learning backbone
from tensorflow.keras.applications import MobileNetV2

# Model architecture layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

# Training callbacks for optimization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# -------------------------------
# ⚖️ Class Imbalance Handling
# -------------------------------
from sklearn.utils.class_weight import compute_class_weight  # Compute balanced class weights


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 📂 Data Preparation & Augmentation

This section handles the loading, preprocessing, and augmentation of the dataset.

- **Training Set**: Loaded from the `train/` folder, with subfolders representing each class (7 categories).
- **Validation Set**: Automatically split from the training set using `validation_split=0.2` (i.e., 80% training, 20% validation).
- **Test Set**: Contains unlabeled `.jpg` images in the `test/` folder. These are loaded without labels to generate final predictions.

We apply data augmentation techniques such as rotation, zoom, and horizontal flip to improve model generalization, along with pixel normalization (`rescale=1./255`).

The `ImageDataGenerator` class is used to create generators that will feed batches of images to the model during training and evaluation.

In [3]:
# Get the current working directory (assumes the notebook is located at the root of the 'data' folder)
base_dir = os.getcwd()

# Define absolute paths to the training and test directories
train_dir = os.path.join(base_dir, 'data', 'train')
test_dir = os.path.join(base_dir, 'data', 'test')

# -------------------------
# Data Augmentation & Preprocessing
# -------------------------

# Define an ImageDataGenerator for training with real-time data augmentation and validation split
train_datagen = ImageDataGenerator(
    rescale=1./255,              # Normalize pixel values to [0,1]
    rotation_range=20,           # Randomly rotate images by up to 20 degrees
    zoom_range=0.2,              # Random zoom within the range [1.0 - 1.2]
    horizontal_flip=True,        # Random horizontal flipping
    validation_split=0.2         # Reserve 20% of data for validation
)

# Training data generator (80% of the training set)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),      # Resize all images to 224x224
    batch_size=32,
    class_mode='categorical',    # Use categorical labels (one-hot encoded)
    subset='training',           # This subset is used for training
    shuffle=True                 # Shuffle the data for better generalization
)

# Validation data generator (20% of the training set)
val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',         # This subset is used for validation
    shuffle=False                # Do not shuffle to preserve order for evaluation
)

# -------------------------
# Test Data Preprocessing
# -------------------------

# Define a separate generator for test data (unlabeled)
# All test images are expected to be directly inside the "test" folder without subdirectories
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    directory=os.path.join(base_dir, 'data'),  # Points to the parent of 'test'
    classes=['test'],                          # Treats 'test/' as a single pseudo-class folder
    target_size=(224, 224),
    batch_size=1,
    class_mode=None,                           # No labels provided for test images
    shuffle=False                              # Keep filenames in order for submission.csv
)

Found 3821 images belonging to 7 classes.
Found 951 images belonging to 7 classes.
Found 504 images belonging to 1 classes.


## 🧠 Model Architecture – Transfer Learning with MobileNetV2

In this section, we build a **convolutional neural network** using **transfer learning**.  
We utilize **MobileNetV2**, a lightweight and efficient model pretrained on **ImageNet**.

### 🔧 Architecture Overview
- The **top classification head is removed** (`include_top=False`).
- A **custom classification head** is added on top of the base model.

### 🏗️ Custom Head Components:
1. **GlobalAveragePooling2D**  
   Reduces the spatial dimensions of feature maps, converting them into a single feature vector per sample.

2. **Dropout (50%)**  
   Adds regularization to prevent overfitting by randomly deactivating 50% of neurons during training.

3. **Dense Layer (Softmax, 7 classes)**  
   Outputs probabilities across **7 tea leaf disease categories**.

### 🚫 Freezing the Base Model
Initially, **all layers of MobileNetV2 are frozen** (`trainable = False`).  
This ensures that only the newly added classification head is trained during the first phase.

### 🧪 Model Compilation
The model is compiled with:
- **Adam optimizer** with a learning rate of `1e-4`
- **Categorical crossentropy** as the loss function (for multi-class classification)
- **Accuracy** as the evaluation metric


In [4]:
# Load MobileNetV2 without the classification head
base_model_mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Add custom head
x = base_model_mobilenet.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
predictions = Dense(7, activation='softmax')(x)

model_mobilenet = Model(inputs=base_model_mobilenet.input, outputs=predictions)

# Freeze base model layers initially
for layer in base_model_mobilenet.layers:
    layer.trainable = False

# Compile the model
model_mobilenet.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])


## ⚖️ Class Weighting to Handle Dataset Imbalance

The TeaLeafBD dataset exhibits a noticeable **class imbalance**, with some disease categories significantly under-represented.  
If not addressed, this imbalance can lead the model to **favor majority classes**, resulting in poor generalization and biased predictions.

To mitigate this, we apply **class weighting**, a technique that adjusts the training loss function to:

- Assign **higher penalties to errors** made on under-represented classes
- **Reduce bias** toward majority classes
- Encourage the model to **learn all classes fairly**

The weights are computed using `sklearn.utils.class_weight.compute_class_weight` in `'balanced'` mode, which automatically assigns weights inversely proportional to class frequencies.

In [5]:
print(Counter(train_generator.classes))

# Compute class weights to address dataset imbalance
# 'balanced' mode assigns higher weights to under-represented classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),  # The set of class indices
    y=train_generator.classes                    # The actual class labels in the training data
)

# Convert weights to dictionary format expected by model.fit()
class_weights_dict = dict(enumerate(class_weights))

# Display the computed weights for each class index
print("Class Weights:", class_weights_dict)

Counter({5: 948, 2: 763, 6: 659, 3: 422, 1: 373, 4: 345, 0: 311})
Class Weights: {0: 1.7551676619200736, 1: 1.4634239754883187, 2: 0.7154090994195843, 3: 1.2935003385240351, 4: 1.5821946169772256, 5: 0.5757986738999398, 6: 0.8283112941686538}


## 🏁 Phase 1 – Training the Custom Classification Head (MobileNetV2 Frozen)

In this block, we **train only the custom head** on top of the frozen MobileNetV2 base model.

### 🔧 Callbacks for Training
- **EarlyStopping**: Stops training early if validation loss does not improve for 3 consecutive epochs. Restores the best model weights.
- **ReduceLROnPlateau**: Reduces learning rate by a factor of 0.5 if validation loss stagnates for 2 epochs.
- **ModelCheckpoint**: Saves the model with the best validation loss during training (`best_model_mobilenet.keras`).

### 🏋️ Training Parameters
- Only the custom classification head is trained.
- Uses **class_weight** to compensate for class imbalance.
- Runs for **20 epochs**, with real-time validation monitoring.

In [1]:
# Callbacks
checkpoint_mobilenet = ModelCheckpoint('best_model_mobilenet.keras', monitor='val_loss', save_best_only=True, verbose=1)

# Train head only
history_mobilenet = model_mobilenet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
        checkpoint_mobilenet
    ],
    class_weight=class_weights_dict
)

NameError: name 'ModelCheckpoint' is not defined

## 🛠️ Phase 2 – Fine-Tuning the Top Layers of MobileNetV2

After training the custom head, we proceed to **fine-tune the last 30 layers** of MobileNetV2.

### 🔓 Unfreezing Layers
- The last **30 layers** of the base model are unfrozen and made trainable to adapt pre-learned features to our specific dataset.

### ⚙️ Recompiling the Model
- Uses a **smaller learning rate (`1e-5`)** to avoid destroying learned features.
- Loss and metric functions remain the same.

### 🏋️ Fine-Tuning Parameters
- Fine-tunes the newly unfrozen layers.
- Runs for **20 epochs** with early stopping and learning rate adjustment.
- Uses **class weighting** to balance the dataset.

In [ ]:
# Fine-tune last 30 layers (optional Phase 2 for MobileNetV2)
for layer in base_model_mobilenet.layers[-30:]:
    layer.trainable = True

model_mobilenet.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

# Fine-tune the unfrozen layers
history_mobilenet_ft = model_mobilenet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
    ],
    class_weight=class_weights_dict
)

## 🧪 Data Augmentation – Enhanced Transformations for Robustness

In this block, we define **advanced data augmentation** strategies to artificially increase dataset diversity and improve model generalization.

### 🔄 Data Augmentation Techniques
- **Rotation (45°)**: Stronger rotations to simulate leaf orientation.
- **Zoom (30%)**: Aggressive zoom for focusing on leaf details.
- **Shear Transformations (0.2)**: Geometric distortions to simulate deformations.
- **Horizontal & Vertical Flips**: Critical for leaves which can appear in any orientation.
- **Brightness Variations (0.7 to 1.3)**: Simulates different lighting conditions.

### 🔄 Data Generators
- Creates **training** and **validation** generators from the dataset directory.
- Uses a **20% validation split**.
- Training data is **shuffled**, validation is kept consistent.

In [ ]:
# Data Augmentation avancée (boostée)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,       # + fort
    zoom_range=0.3,          # gros zoom
    shear_range=0.2,         # déformation
    horizontal_flip=True,
    vertical_flip=True,      # flip vertical important pour les feuilles
    brightness_range=[0.7, 1.3],  # variations fortes de lumière
    validation_split=0.2
)

# Training & Validation generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

## 🏆 Phase 3 – Full Model Fine-Tuning (Entire MobileNetV2)

In this final phase, we fine-tune **all layers of MobileNetV2**, allowing the model to fully adapt to the tea leaf disease classification task.

### 📥 Loading Best Weights
- Loads the best model obtained from previous training phase (`best_model_mobilenet.keras`).

### 🔓 Unfreezing All Layers
- Sets **all layers as trainable** to enable full model fine-tuning.

### ⚙️ Recompiling the Model
- Uses a **very small learning rate (`1e-6`)** to carefully update the entire model without catastrophic forgetting.
- Loss function remains **categorical crossentropy**.

### 🔧 Fine-Tuning Callbacks
- **EarlyStopping**: More patience (5 epochs) for deeper fine-tuning.
- **ReduceLROnPlateau**: Learning rate reduction if validation loss stagnates.
- **ModelCheckpoint**: Saves the best model during this phase (`best_model_mobilenet_phase3.keras`).

### 🏋️ Fine-Tuning Parameters
- Runs for **20 epochs**.
- Continues using **class weighting**.
- Focuses on stabilizing the fine-tuning of the entire network.

In [ ]:
# Load best weights from previous phase
model_mobilenet.load_weights('best_model_mobilenet.keras')

# Unfreeze ALL layers for Phase 3 fine-tuning
for layer in model_mobilenet.layers:
    layer.trainable = True

# Compile with Focal Loss and very small learning rate
model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-6),
    loss="categorical_crossentropy",
    metrics=['accuracy']
)


# Callbacks to stabilize fine-tuning
checkpoint_phase3 = ModelCheckpoint(
    'best_model_mobilenet_phase3.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

fine_tune_callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
    checkpoint_phase3
]

# Fine-tune entire MobileNetV2 model
history_mobilenet_da = model_mobilenet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=fine_tune_callbacks,
    class_weight=class_weights_dict
)

## 📤 Prediction & Submission

With the fine-tuned model trained, we now generate predictions on the **unlabeled test set**.

The results are mapped back to class labels and saved in the required `submission.csv` format:

In [ ]:
# Predict using the fine-tuned model
preds = model_mobilenet.predict(test_generator, verbose=1)
pred_classes = np.argmax(preds, axis=1)

# Map predicted class indices back to class names
label_map = {v: k for k, v in train_generator.class_indices.items()}
filenames = [os.path.basename(f) for f in test_generator.filenames]
submission = pd.DataFrame({
    'filename': filenames,
    'class': [label_map[k] for k in pred_classes]
})

# Export the predictions to a CSV file
submission.to_csv('submission.csv', index=False)

## 📈 Training Progress Visualization

To monitor the model’s learning process, we visualize training and validation performance over each epoch.

This helps assess:
- Convergence speed
- Overfitting or underfitting
- Improvement brought by fine-tuning

In [ ]:
# Function to plot training and validation curves for multiple training phases
def plot_training_history(history_phase1, history_phase2=None, history_phase3=None, title_suffix=''):
    # Extract metrics from phase 1
    acc = history_phase1.history['accuracy']
    val_acc = history_phase1.history['val_accuracy']
    loss = history_phase1.history['loss']
    val_loss = history_phase1.history['val_loss']

    # Append metrics from phase 2 if available
    if history_phase2:
        acc += history_phase2.history['accuracy']
        val_acc += history_phase2.history['val_accuracy']
        loss += history_phase2.history['loss']
        val_loss += history_phase2.history['val_loss']

    # Append metrics from phase 3 if available
    if history_phase3:
        acc += history_phase3.history['accuracy']
        val_acc += history_phase3.history['val_accuracy']
        loss += history_phase3.history['loss']
        val_loss += history_phase3.history['val_loss']

    epochs_range = range(1, len(acc) + 1)

    # Plot Accuracy
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.title('Training and Validation Accuracy' + title_suffix)
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.title('Training and Validation Loss' + title_suffix)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()
    
# Call the function with all three training histories
plot_training_history(history_mobilenet, history_mobilenet_ft, history_mobilenet_da)